# Text Cleaning, Lemmatization, Stop Words Removal, Label Encoding, and TF-IDF

This notebook demonstrates:
- Text cleaning
- Stop words removal
- Lemmatization
- Label encoding
- TF-IDF representation

In [4]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Sample labeled data
raw_df = pd.DataFrame({
    'text': [
        'I absolutely loved the movie, it was fantastic and inspiring!',
        'The service was slow and the food was disappointing.',
        'This product works well and saves a lot of time.',
        'The app crashed twice and the experience was frustrating.',
        'Customer support was helpful and resolved my issue quickly.',
    ],
    'label': ['positive', 'negative', 'positive', 'negative', 'positive']
})

print('Original data:')
print(raw_df)

Original data:
                                                text     label
0  I absolutely loved the movie, it was fantastic...  positive
1  The service was slow and the food was disappoi...  negative
2   This product works well and saves a lot of time.  positive
3  The app crashed twice and the experience was f...  negative
4  Customer support was helpful and resolved my i...  positive


In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [word for word in text.split() if word not in stop_words]
    lemmas = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(lemmas)

raw_df['cleaned_text'] = raw_df['text'].apply(clean_text)

label_encoder = LabelEncoder()
raw_df['label_encoded'] = label_encoder.fit_transform(raw_df['label'])

print('Processed data:')
print(raw_df)

label_mapping = {label: int(code) for code, label in enumerate(label_encoder.classes_)}
print('\nLabel encoding mapping:')
print(label_mapping)

Processed data:
                                                text     label  \
0  I absolutely loved the movie, it was fantastic...  positive   
1  The service was slow and the food was disappoi...  negative   
2   This product works well and saves a lot of time.  positive   
3  The app crashed twice and the experience was f...  negative   
4  Customer support was helpful and resolved my i...  positive   

                                      cleaned_text  label_encoded  
0       absolutely loved movie fantastic inspiring              1  
1                  service slow food disappointing              0  
2                  product work well save lot time              1  
3         app crashed twice experience frustrating              0  
4  customer support helpful resolved issue quickly              1  

Label encoding mapping:
{'negative': 0, 'positive': 1}


In [6]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(raw_df['cleaned_text'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f'Doc {i}' for i in range(1, len(raw_df) + 1)]
)

print('TF-IDF Representation:')
print(tfidf_df.round(3))

print('\nNotebook-only output complete.')

TF-IDF Representation:
       absolutely    app  crashed  customer  disappointing  experience  \
Doc 1       0.447  0.000    0.000     0.000            0.0       0.000   
Doc 2       0.000  0.000    0.000     0.000            0.5       0.000   
Doc 3       0.000  0.000    0.000     0.000            0.0       0.000   
Doc 4       0.000  0.447    0.447     0.000            0.0       0.447   
Doc 5       0.000  0.000    0.000     0.408            0.0       0.000   

       fantastic  food  frustrating  helpful  ...  quickly  resolved   save  \
Doc 1      0.447   0.0        0.000    0.000  ...    0.000     0.000  0.000   
Doc 2      0.000   0.5        0.000    0.000  ...    0.000     0.000  0.000   
Doc 3      0.000   0.0        0.000    0.000  ...    0.000     0.000  0.408   
Doc 4      0.000   0.0        0.447    0.000  ...    0.000     0.000  0.000   
Doc 5      0.000   0.0        0.000    0.408  ...    0.408     0.408  0.000   

       service  slow  support   time  twice   well   work